## Test with data – the real backend on the real database

Runs the **implemented** propose logic ([Tools/ProposeLogic.py](../../Tools/ProposeLogic.py)) on `Server/server_data/*.yaml` and shows every intermediate value per recipe:
date_score → reward_score → damping → total score → pool → draw probability.

The formulas themselves are derived in the other notebooks ([date_score](date_score.ipynb), [reward_score](reward_score.ipynb), [damping](damping.ipynb), [total_score](total_score.ipynb)) – each of them checks itself against the backend.

Feed the database (website: create / cook / rate recipes), then just **re-run all cells**. Nothing is written back.

In [ ]:
import sys
import random
from collections import Counter
from datetime import date
from pathlib import Path

import pandas as pd

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Tools").is_dir())
sys.path.insert(0, str(root))
from Tools import ProposeLogic as pl
from Tools import YamlEditor as yr
from Server import smartfood_server as srv

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 250)

TODAY = date.today()      # oder fest, z.B. date(2026, 10, 22)
USER = None               # user_id fuer Wunschliste / Blacklist, z.B. "1"
MEAL_TYPE, DIET = "main_dish", "any"
T = pl.SOFTMAX_T

data = srv.load_all()
print(f"{len(data['recipes'])} recipes, {len(data['ingredients'])} ingredients, "
      f"{len(data['history'])} history entries, today = {TODAY}")
for problem in yr.data_problems(data):
    print("PROBLEM:", problem)

### App values (rotation point)
`count_usage_app` = distinct cook days in the last 30 days (0 → 3), `count_recipes_twice` = recipes with ≥ 2 dates (at least 5), `periodcycle_app` at least 7 days.

In [ ]:
app = pl.app_stats(data, TODAY)
pd.Series(app).round(2).to_frame("value")

### last_supper (damping reference)
The last 3 meals cooked within 4 days, merged into one list.

In [ ]:
by_id = {r["recipe_id"]: r for r in data["recipes"]}
ls_ids = pl.last_supper(data, TODAY)
print("last_supper:", [by_id[i]["name"] for i in ls_ids if i in by_id] or "-")
ls = pl.merge_last_supper([pl.recipe_features(by_id[i], data) for i in ls_ids if i in by_id])
print("season now:", pl.current_season(TODAY))
ls

### Every recipe, every score

| Column | Meaning |
|---|---|
| `pool` | `main` (cooked or wishlist), `new` (never cooked), or why the hard filter removed it |
| `n_dates`, `dist_last`, `mean`, `std`, `repeat` | inputs of date_score |
| `T`, `amp` | target distance and amplification (≤ 2 dates → `amp = 1`, no `T`) |
| `taste`, `effort`, `minutes` | inputs of reward_score (missing ratings → 5) |
| `mult`, `offset`, `damping` | damping and the rules that fired |
| `score` | total score (`median` = never cooked without times) |
| `p` | probability of being drawn **first** from its pool at `T` |

In [ ]:
ds = pl.build_propose_dataset(data, USER, TODAY)
blacklist = pl.user_blacklist(data, USER)
candidates = pl.hard_filter(ds, ls_ids, MEAL_TYPE, DIET, blacklist)
main_pool, new_pool = pl.split_pools(ds, candidates) if candidates else ({}, {})
ls_roots = {pl.variant_root(by_id[i]) for i in ls_ids if i in by_id}

def filter_reason(rid):
    d, r = ds[rid], ds[rid]["recipe"]
    if r.get("recipe_type") != MEAL_TYPE or not pl.diet_ok(r.get("veg_type"), DIET):
        return f"filtered: type {r.get('recipe_type')}"
    if d["remove"] and not d["wishlist"]:
        return "filtered: remove (cooked <= 5 days)"
    if pl.variant_root(r) in ls_roots:
        return "filtered: last_supper family"
    if rid in blacklist:
        return "filtered: blacklist"
    if pl.is_same_day(r):
        return "filtered: > 60 min"
    return "filtered"

def probs(pool):
    ids = list(pool)
    return dict(zip(ids, pl.softmax([pool[i] for i in ids], T))) if ids else {}

p_main, p_new = probs(main_pool), probs(new_pool)
rows = []
for rid, d in ds.items():
    dt, rw = d["date"], d["reward"]
    pool = "main" if rid in main_pool else "new" if rid in new_pool else filter_reason(rid)
    score = main_pool.get(rid, new_pool.get(rid, d["score"]))
    rows.append({
        "id": rid, "name": d["recipe"].get("name"), "pool": pool,
        "n_dates": dt["n_dates"], "dist_last": dt["dist_last"], "mean": dt["mean"], "std": dt["std"],
        "repeat": dt["repeat"], "T": dt["target"], "amp": dt["amp"],
        "date_score": d["date_score"], "remove": d["remove"],
        "taste": rw["taste"], "effort": rw["effort"], "minutes": rw["minutes"], "reward": d["reward_score"],
        "mult": d["damp_score_mult"], "offset": d["damp_score_offset"], "damping": ", ".join(d["damp_rules"]),
        "wishlist": d["wishlist"],
        "score": score if d["score"] is not None or score is None else f"{score:.2f} (median)",
        "p": p_main.get(rid, p_new.get(rid)),
    })
table = pd.DataFrame(rows).sort_values(["pool", "p"], ascending=[True, False])
print(f"main pool: {len(main_pool)}, new pool: {len(new_pool)}, filtered: {len(ds) - len(candidates)}")
table.round(2)

### Proposals
One real `propose()` call (like the website), the "Neue Rezepte" button, and how often each recipe lands in a list over many runs.

In [ ]:
rng = random.Random(1)
kw = dict(user_id=USER, today=TODAY, meal_type=MEAL_TYPE, diet=DIET, t=T)
print("propose():       ", [r["name"] for r in pl.propose(data, rng=rng, **kw)] or "nothing -> message on the website")
print("Neue Rezepte:    ", [r["name"] for r in pl.propose(data, rng=rng, pool="new", **kw)] or "-")

RUNS = 1000
hits = Counter(r["recipe_id"] for _ in range(RUNS) for r in pl.propose(data, rng=rng, **kw))
pd.DataFrame([{"name": by_id[i]["name"], "in list": f"{n / RUNS:.1%}"} for i, n in hits.most_common()])